In [1]:
from pathlib import Path
from miditok import REMI

TOKENIZER_PATH = Path("../configs/tokenizer.json")

tokenizer = REMI(params=TOKENIZER_PATH)

print("Vocabulary size:", len(tokenizer))

Vocabulary size: 20000


c:\Users\aqdas\Minor_Project\piano-music-composition\.venv\Lib\site-packages\miditok\classes.py:886: UserWarning: Argument nb_tempos has been renamed num_tempos, you should consider to updateyour code with this new argument name.
  return cls(**input_dict, **kwargs)


In [2]:
# from pathlib import Path

MAESTRO_DIR = Path("../data/raw/maestro")

midi_files = list(MAESTRO_DIR.rglob("*.midi"))

print("Total MIDI files:", len(midi_files))
print(midi_files[:3])

Total MIDI files: 1276
[WindowsPath('../data/raw/maestro/2004/MIDI-Unprocessed_SMF_02_R1_2004_01-05_ORIG_MID--AUDIO_02_R1_2004_05_Track05_wav.midi'), WindowsPath('../data/raw/maestro/2004/MIDI-Unprocessed_SMF_02_R1_2004_01-05_ORIG_MID--AUDIO_02_R1_2004_06_Track06_wav.midi'), WindowsPath('../data/raw/maestro/2004/MIDI-Unprocessed_SMF_02_R1_2004_01-05_ORIG_MID--AUDIO_02_R1_2004_08_Track08_wav.midi')]


In [3]:
midi_path = midi_files[0]

tokens = tokenizer(midi_path)
token_ids = tokens[0].ids

print("Token count:", len(token_ids))
print("First 20 IDs:", token_ids[:20])

Token count: 11151
First 20 IDs: [1482, 16001, 1123, 151, 2177, 153, 1151, 16659, 159, 267, 958, 175, 4855, 452, 131, 5990, 316, 2079, 133, 19222]


In [4]:
print("Max token ID:", max(token_ids))
print("Min token ID:", min(token_ids))

Max token ID: 19999
Min token ID: 5


In [5]:
CONTEXT_LENGTH = 2048
STRIDE = 1024

chunks = []

for start in range(0, len(token_ids) - CONTEXT_LENGTH + 1, STRIDE):
    chunk = token_ids[start:start + CONTEXT_LENGTH]
    chunks.append(chunk)

print("Number of chunks:", len(chunks))
print("Chunk length:", len(chunks[0]))
print("First chunk:", chunks[0][:10])
print("Last chunk:", chunks[-1][-10:])

Number of chunks: 9
Chunk length: 2048
First chunk: [1482, 16001, 1123, 151, 2177, 153, 1151, 16659, 159, 267]
Last chunk: [19830, 540, 454, 155, 504, 11196, 194, 928, 134, 806]


In [6]:
CONTEXT_LENGTH = 2048
STRIDE = 1024

total_chunks = 0
total_tokens = 0

for midi_path in midi_files:
    tokens = tokenizer(midi_path)
    token_ids = tokens[0].ids

    total_tokens += len(token_ids)

    if len(token_ids) >= CONTEXT_LENGTH:
        num_chunks = (
            (len(token_ids) - CONTEXT_LENGTH) // STRIDE
        ) + 1
        total_chunks += num_chunks

print("Training MIDI files:", len(midi_files))
print("Total tokens:", total_tokens)
print("Total chunks:", total_chunks)

Training MIDI files: 1276
Total tokens: 9830040
Total chunks: 7687


In [4]:
import pandas as pd

csv_path = Path('../data/maestro-v3.0.0.csv')

metadata = pd.read_csv(csv_path)

print(metadata.columns.tolist())
print(metadata["split"].value_counts())

['canonical_composer', 'canonical_title', 'split', 'year', 'midi_filename', 'audio_filename', 'duration']
split
train         962
test          177
validation    137
Name: count, dtype: int64


In [5]:
# from pathlib import Path

MAESTRO_DIR = Path("../data/raw/maestro")

train_midi_files = [
    MAESTRO_DIR / path
    for path in metadata.loc[metadata["split"] == "train", "midi_filename"]
]

val_midi_files = [
    MAESTRO_DIR / path
    for path in metadata.loc[metadata["split"] == "validation", "midi_filename"]
]

test_midi_files = [
    MAESTRO_DIR / path
    for path in metadata.loc[metadata["split"] == "test", "midi_filename"]
]

print("Train:", len(train_midi_files))
print("Validation:", len(val_midi_files))
print("Test:", len(test_midi_files))

print("\nExample:")
print(train_midi_files[0])
print("Exists:", train_midi_files[0].exists())

Train: 962
Validation: 137
Test: 177

Example:
..\data\raw\maestro\2018\MIDI-Unprocessed_Chamber3_MID--AUDIO_10_R3_2018_wav--1.midi
Exists: True


In [9]:
CONTEXT_LENGTH = 2048
STRIDE = 1024

total_chunks = 0
total_tokens = 0
files_with_chunks = 0

for midi_path in train_midi_files:
    tokens = tokenizer(midi_path)
    token_ids = tokens[0].ids

    total_tokens += len(token_ids)

    if len(token_ids) >= CONTEXT_LENGTH:
        num_chunks = (
            (len(token_ids) - CONTEXT_LENGTH) // STRIDE
        ) + 1

        total_chunks += num_chunks
        files_with_chunks += 1

print("Training files:", len(train_midi_files))
print("Files producing chunks:", files_with_chunks)
print("Total tokens:", total_tokens)
print("Total chunks:", total_chunks)

Training files: 962
Files producing chunks: 909
Total tokens: 7905631
Total chunks: 6274


In [10]:
from pathlib import Path
import torch

CONTEXT_LENGTH = 2048
STRIDE = 1024

train_chunks = []

for i, midi_path in enumerate(train_midi_files):
    tokens = tokenizer(midi_path)
    token_ids = tokens[0].ids

    for start in range(
        0,
        len(token_ids) - CONTEXT_LENGTH + 1,
        STRIDE
    ):
        chunk = token_ids[start:start + CONTEXT_LENGTH]
        train_chunks.append(chunk)

    if (i + 1) % 100 == 0:
        print(f"Processed {i + 1}/{len(train_midi_files)} files")

print("Total chunks:", len(train_chunks))
print("Chunk length:", len(train_chunks[0]))

Processed 100/962 files
Processed 200/962 files
Processed 300/962 files
Processed 400/962 files
Processed 500/962 files
Processed 600/962 files
Processed 700/962 files
Processed 800/962 files
Processed 900/962 files
Total chunks: 6274
Chunk length: 2048


In [11]:
output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

torch.save(train_chunks, output_dir / "train_chunks.pt")

print("Saved:", output_dir / "train_chunks.pt")

Saved: ..\data\processed\train_chunks.pt


In [7]:
# Validation Chunks creation
# from pathlib import Path
import torch

CONTEXT_LENGTH = 2048
STRIDE = 1024

val_chunks = []

for i, midi_path in enumerate(val_midi_files):
    tokens = tokenizer(midi_path)
    token_ids = tokens[0].ids

    for start in range(
        0,
        len(token_ids) - CONTEXT_LENGTH + 1,
        STRIDE
    ):
        val_chunks.append(
            token_ids[start:start + CONTEXT_LENGTH]
        )

    if (i + 1) % 25 == 0:
        print(f"Processed {i + 1}/{len(val_midi_files)} files")

print("Validation files:", len(val_midi_files))
print("Validation chunks:", len(val_chunks))
print("Chunk length:", len(val_chunks[0]) if val_chunks else 0)

output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

torch.save(val_chunks, output_dir / "val_chunks.pt")

print("Saved validation chunks.")

Processed 25/137 files
Processed 50/137 files
Processed 75/137 files
Processed 100/137 files
Processed 125/137 files
Validation files: 137
Validation chunks: 682
Chunk length: 2048
Saved validation chunks.
